In [16]:
import importlib
import pandas as pd
import sys
from pathlib import Path
import os

root = Path.cwd().parent.parent
sys.path.insert(0, str(root / "code"))

import data.load_data as load_data
import data.labels as labels

importlib.reload(load_data)
importlib.reload(labels)

from data.load_data import load_alerts_from_json
from data.labels import assign_labels

In [10]:
output_file = "combined_ait_ads"
dir_path = os.path.join(root, "data/ait_ads")

### Load data
##### Wazuh + Aminer Alerts (AIT-ADS, .json format) to .parquet

In [ ]:
df = load_alerts_from_json(output_file, dir_path)

##### Load .parquet to notebook

In [11]:
df = pd.read_parquet(
    os.path.join(dir_path, f"{output_file}.parquet"),
    engine="pyarrow"
)

In [11]:
df.columns

Index(['timestamp', 'scenario', 'source', 'category', 'entity', 'raw_log',
       'host_ip', 'host', 'rule_id', 'rule_desc', 'groups', 'groups_raw',
       'groups_str', 'alert_channel', 'decoder', 'decoder_parent', 'location',
       'mitre_ids', 'mitre_tactic', 'mitre_technique', 'username', 'procname',
       'aminer_component_type', 'aminer_training_mode', 'aminer_new_event',
       'aminer_component_name', 'aminer_message', 'aminer_persistence_file',
       'aminer_affected_paths', 'aminer_affected_values',
       'aminer_log_resources', 'aminer_log_lines_count', 'srcip', 'dstip',
       'srcport', 'dstport', 'proto', 'is_ids_alert', 'ids_signature',
       'ids_category', 'ids_severity', 'data_json', 'agent_id', 'agent_name',
       'manager_name', 'rule_firedtimes', 'rule_frequency', 'rule_mail',
       'wazuh_level', 'rule_groups_json', 'rule_pci_dss', 'rule_gdpr',
       'rule_hipaa', 'rule_nist_800_53', 'rule_tsc', 'log_source_path',
       'input_type', 'predecoder_timestamp

In [5]:
# check for duplicate alert_ids
duplicate_alert_ids = df[df.duplicated(subset=["alert_id"], keep=False)]
print(f"Number of duplicate alert_ids: {len(duplicate_alert_ids)}")

Number of duplicate alert_ids: 0


#### Sanity checks

In [6]:
print("Min timestamp in dataset: ", df["timestamp"].min(), "\nMax timestamp in dataset: ", df["timestamp"].max())

Min timestamp in dataset:  2022-01-14 00:00:01.750000+00:00 
Max timestamp in dataset:  2022-02-08 23:59:46.130000+00:00


## Labelling

In [17]:
labeled_df = assign_labels(df, dir_path)

Labeling dataset...
Reading labels from: /Users/annavisman/stack/TUDelft/thesis/msc-thesis/data/ait_ads/labels.csv
Done. Wrote labeled dataset to: /Users/annavisman/stack/TUDelft/thesis/msc-thesis/data/ait_ads/labeled_combined_ait_ads.parquet


### Attack report

In [18]:
labeled_df = pd.read_parquet(
    os.path.join(dir_path, f"labeled_{output_file}.parquet"),
    engine="pyarrow"
)

In [20]:
labeled_df.columns

Index(['timestamp', 'scenario', 'source', 'category', 'entity', 'raw_log',
       'host_ip', 'host', 'rule_id', 'rule_desc', 'groups', 'groups_raw',
       'groups_str', 'alert_channel', 'decoder', 'decoder_parent', 'location',
       'mitre_ids', 'mitre_tactic', 'mitre_technique', 'username', 'procname',
       'aminer_component_type', 'aminer_training_mode', 'aminer_new_event',
       'aminer_component_name', 'aminer_message', 'aminer_persistence_file',
       'aminer_affected_paths', 'aminer_affected_values',
       'aminer_log_resources', 'aminer_log_lines_count', 'srcip', 'dstip',
       'srcport', 'dstport', 'proto', 'is_ids_alert', 'ids_signature',
       'ids_category', 'ids_severity', 'data_json', 'agent_id', 'agent_name',
       'manager_name', 'rule_firedtimes', 'rule_frequency', 'rule_mail',
       'wazuh_level', 'rule_groups_json', 'rule_pci_dss', 'rule_gdpr',
       'rule_hipaa', 'rule_nist_800_53', 'rule_tsc', 'log_source_path',
       'input_type', 'predecoder_timestamp

In [ ]:
# number of attakcs in the dataset for each scenario
print(labeled_df["y"].value_counts())

labeled_df[labeled_df["y"] == 1]["scenario"].value_counts()
# or
labeled_df.groupby("scenario")["y"].sum().sort_values(ascending=False)

print(labeled_df["event_label"].value_counts())
print(labeled_df.groupby("scenario")["event_label"].value_counts())

overall, by_scn, summary = attacks_per_period_report(
    labeled_df,
    period="day",                  # "hour" | "day" | "week"
    scenario_col="scenario",
    tz="UTC",
    week_start="MON",
)

print(summary)
# display(overall.head(10))
display(by_scn[by_scn["has_both_classes"]].head(50))

y
0    2650506
1       5315
Name: count, dtype: int64
event_label
benign                         2349042
attack:dirb                     249226
attack:wpscan                    54570
attack:cracking                   1987
attack:service_scans               629
attack:dnsteal                     240
attack:privilege_escalation        108
attack:webshell                     18
attack:reverse_shell                 1
Name: count, dtype: int64
scenario         event_label                
fox              benign                         401076
                 attack:dirb                     62305
                 attack:wpscan                    9474
                 attack:cracking                   168
                 attack:service_scans               63
                 attack:dnsteal                      9
                 attack:privilege_escalation         7
                 attack:webshell                     2
harrison         benign                         521060
                 

,bucket,n_total,n_attacks,n_benign,attack_rate,has_both_classes
0,2022-01-14 00:00:00+00:00,32056,0,32056,0.000000,False
1,2022-01-15 00:00:00+00:00,36752,0,36752,0.000000,False
2,2022-01-16 00:00:00+00:00,20477,4,20473,0.000195,True
3,2022-01-17 00:00:00+00:00,12759,169,12590,0.013246,True
4,2022-01-18 00:00:00+00:00,5659,898,4761,0.158685,True
5,2022-01-19 00:00:00+00:00,17520,0,17520,0.000000,False
6,2022-01-20 00:00:00+00:00,6581,0,6581,0.000000,False
7,2022-01-21 00:00:00+00:00,18912,0,18912,0.000000,False
8,2022-01-22 00:00:00+00:00,12489,0,12489,0.000000,False
9,2022-01-23 00:00:00+00:00,13147,48,13099,0.003651,True


,scenario,bucket,n_total,n_attacks,n_benign,attack_rate,has_both_classes
2,fox,2022-01-17 00:00:00+00:00,3843,2,3841,0.000520,True
3,fox,2022-01-18 00:00:00+00:00,5659,898,4761,0.158685,True
9,harrison,2022-02-08 00:00:00+00:00,19227,1552,17675,0.080720,True
13,russellmitchell,2022-01-24 00:00:00+00:00,4636,100,4536,0.021570,True
16,santos,2022-01-16 00:00:00+00:00,14997,4,14993,0.000267,True
17,santos,2022-01-17 00:00:00+00:00,8916,167,8749,0.018730,True
22,shaw,2022-01-29 00:00:00+00:00,4211,21,4190,0.004987,True
28,wardbeck,2022-01-23 00:00:00+00:00,7669,48,7621,0.006259,True
32,wheeler,2022-01-29 00:00:00+00:00,28238,226,28012,0.008003,True
33,wheeler,2022-01-30 00:00:00+00:00,29731,1347,28384,0.045306,True


Check start and end times per scenario

In [21]:
df = labeled_df.copy()

df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True, errors="coerce")

# scenario timeline (start/end, duration, counts)
timeline = (
    df.dropna(subset=["timestamp"])
      .groupby("scenario", sort=False)
      .agg(
          start=("timestamp", "min"),
          end=("timestamp", "max"),
          duration=("timestamp", lambda s: s.max() - s.min()),
          n_events=("timestamp", "size"),
          n_attack=("y", "sum"),
          first_attack=("timestamp", lambda s: s[df.loc[s.index, "y"].eq(1)].min()),
          last_attack=("timestamp",  lambda s: s[df.loc[s.index, "y"].eq(1)].max()),
      )
      .sort_values("start")
)

print(timeline)

# check if scenarios can be chained without overlap (sorted by start)
chain_check = timeline.reset_index().sort_values("start")
chain_check["gap_from_prev"] = chain_check["start"] - chain_check["end"].shift(1)
chain_check["overlaps_prev"] = chain_check["gap_from_prev"] < pd.Timedelta(0)

print(chain_check[["scenario", "start", "end", "gap_from_prev", "overlaps_prev"]])

                                           start  \
scenario                                           
santos          2022-01-14 00:00:01.750000+00:00   
fox             2022-01-15 00:00:01.160000+00:00   
wardbeck               2022-01-19 00:00:01+00:00   
russellmitchell        2022-01-21 00:00:01+00:00   
shaw                   2022-01-25 00:00:01+00:00   
wheeler                2022-01-26 00:00:01+00:00   
wilson                 2022-02-03 00:00:01+00:00   
harrison        2022-02-04 00:00:01.880000+00:00   

                                             end               duration  \
scenario                                                                  
santos                 2022-01-17 23:51:42+00:00 3 days 23:51:40.250000   
fox                    2022-01-19 23:55:36+00:00 4 days 23:55:34.840000   
wardbeck        2022-01-23 23:59:19.170000+00:00 4 days 23:59:18.170000   
russellmitchell        2022-01-24 23:39:13+00:00        3 days 23:39:12   
shaw                   2022-0